# 21 — Risky Bonds & CVA

- **Risky bond pricing** (credit-adjusted NPV)
- **CVA/DVA for interest rate swaps** (Brigo-Masetti)
- Bilateral CVA
- AD: sensitivity of CVA to credit spread

In [ ]:
BACKEND = "cpu"

import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "..") if not os.getcwd().endswith("ql-jax") else ".")
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))

from notebooks._common import setup_backend, load_quantlib, compare_table, timed_ms
jax = setup_backend(BACKEND)
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

---
## Part A: Risky Bond Pricing

In [ ]:
from ql_jax.engines.bond.risky import risky_bond_npv
from ql_jax.instruments.bond import make_fixed_rate_bond
from ql_jax.time.date import Date

today = Date(15, 6, 2024)
maturity = Date(15, 6, 2034)

bond = make_fixed_rate_bond(
    face_value=100.0, coupon_rate=0.05,
    issue_date=today, maturity_date=maturity,
    settlement_days=0, frequency=2)

DISC_RATE = 0.03
HAZARD = 0.02
RECOVERY = 0.40

disc_fn = lambda t: jnp.exp(-DISC_RATE * t)
surv_fn = lambda t: jnp.exp(-HAZARD * t)

In [ ]:
risky_npv = float(risky_bond_npv(bond, disc_fn, surv_fn, RECOVERY))

# Risk-free NPV for comparison
riskfree_npv = float(risky_bond_npv(bond, disc_fn, lambda t: jnp.ones_like(t), 0.0))

credit_adjustment = riskfree_npv - risky_npv

print(f"Risk-free bond NPV  : {riskfree_npv:.6f}")
print(f"Risky bond NPV      : {risky_npv:.6f}")
print(f"Credit adjustment   : {credit_adjustment:.4f} ({credit_adjustment/riskfree_npv*100:.1f}%)")

In [ ]:
# NPV vs hazard rate
hazards = jnp.linspace(0.0, 0.10, 50)
npvs = [float(risky_bond_npv(bond, disc_fn, lambda t, h=h: jnp.exp(-h*t), RECOVERY))
        for h in hazards]

plt.figure(figsize=(8, 5))
plt.plot(np.array(hazards)*10000, npvs, 'b-', linewidth=2)
plt.axhline(riskfree_npv, color='gray', ls='--', label='Risk-free')
plt.xlabel('Hazard Rate (bp)')
plt.ylabel('Bond NPV')
plt.title('Risky Bond NPV vs Hazard Rate')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---
## Part B: CVA for Interest Rate Swaps

In [ ]:
from ql_jax.engines.swap.cva import cva_swap_brigo_masetti

NOTIONAL = 1_000_000.0
SWAP_RATE = 0.04
N_PERIODS = 20         # 5Y swap, quarterly
BLACK_VOL = 0.15
CPTY_HAZARD = 0.02
CPTY_RECOVERY = 0.35

payment_times = [0.25 * (i + 1) for i in range(N_PERIODS)]
accrual_fracs = [0.25] * N_PERIODS

cpty_surv = lambda t: jnp.exp(-CPTY_HAZARD * jnp.float64(t))

In [ ]:
result = cva_swap_brigo_masetti(
    discount_fn=disc_fn, black_vol=BLACK_VOL,
    swap_fixed_rate=SWAP_RATE, payment_times=payment_times,
    accrual_fracs=accrual_fracs, notional=NOTIONAL,
    cpty_survival_fn=cpty_surv, cpty_recovery=CPTY_RECOVERY,
    swap_type=1)

print(f"Base NPV        : {float(result['base_npv']):12.2f}")
print(f"CVA             : {float(result['cva']):12.2f}")
print(f"DVA             : {float(result['dva']):12.2f}")
print(f"Risky NPV       : {float(result['risky_npv']):12.2f}")
print(f"Base fair rate  : {float(result['base_fair_rate'])*10000:8.2f} bp")
print(f"Risky fair rate : {float(result['risky_fair_rate'])*10000:8.2f} bp")
print(f"CVA charge      : {(float(result['risky_fair_rate'])-float(result['base_fair_rate']))*10000:.2f} bp")

---
## 3. CVA vs Counterparty Credit Quality

In [ ]:
hazards_range = np.linspace(0.005, 0.10, 30)
cvas = []
for hz in hazards_range:
    sf = lambda t, h=hz: jnp.exp(-h * jnp.float64(t))
    res = cva_swap_brigo_masetti(
        disc_fn, BLACK_VOL, SWAP_RATE, payment_times, accrual_fracs,
        NOTIONAL, sf, CPTY_RECOVERY, swap_type=1)
    cvas.append(float(res['cva']))

plt.figure(figsize=(8, 5))
plt.plot(hazards_range * 10000, cvas, 'r-', linewidth=2)
plt.xlabel('Counterparty Hazard Rate (bp)')
plt.ylabel('CVA')
plt.title('CVA vs Counterparty Credit Quality')
plt.grid(True, alpha=0.3)
plt.show()

---
## 4. AD: CVA Sensitivity

In [ ]:
def cva_fn(hazard):
    sf = lambda t: jnp.exp(-hazard * jnp.float64(t))
    res = cva_swap_brigo_masetti(
        disc_fn, BLACK_VOL, SWAP_RATE, payment_times, accrual_fracs,
        NOTIONAL, sf, CPTY_RECOVERY, swap_type=1)
    return res['cva']

dcva_dh = float(jax.grad(cva_fn)(CPTY_HAZARD))
print(f"dCVA/d(hazard) : {dcva_dh:12.2f}")
print(f"CVA CS01       : {dcva_dh * 0.0001:12.2f} (per bp of hazard)")

---
## 5. Exercises

1. **Bilateral CVA**: Add investor default risk using `bilateral_cva` and compare to unilateral CVA.
2. **Wrong-way risk**: Correlate the discount rate with the hazard rate and observe the CVA change.
3. **Implied spread**: Given a risky bond price, back out the implied hazard rate using `risky_bond_spread`.

---
*End of Notebook 21*